# Tab 1: judge `ministral-3:8b`

Runs on the 1,000 ClaimDB test claims (`subsets/test-public.jsonl`), one after the other:
1. pipeline 1 (judge writes SQL)
2. pipeline 2 with Arctic-Text2SQL-R1-7B
3. pipeline 2 with OmniSQL-7B
4. pipeline 2 with SLM-SQL-1.5B

**Before running:** Runtime → Change runtime type → **L4 GPU**, and turn on background execution (Pro+).
Run the cells top to bottom. Results go to Google Drive (`MyDrive/sqljudge/results/`), so if the runtime stops,
just run all cells again: finished claims are skipped and the run resumes.

## 1. Settings

In [ ]:
import os
os.environ["JUDGE"] = "ministral-3:8b"          # Ollama tag of the judge
os.environ["NAME"] = "ministral-8b"            # results folder name
os.environ["BUILDERS"] = "a-kore/Arctic-Text2SQL-R1-7B|Arctic-Text2SQL-R1-7B hf.co/mradermacher/OmniSQL-7B-GGUF:Q8_0|OmniSQL-7B hf.co/mradermacher/SLM-SQL-1.5B-GGUF:Q8_0|SLM-SQL-1.5B"   # builder tag|folder name, pipeline 2 runs in this order
os.environ["CLAIMS"] = "subsets/test-public.jsonl"
os.environ["DRIVE"] = "/content/drive/MyDrive/sqljudge"
os.environ["RESULTS"] = "/content/drive/MyDrive/sqljudge/results"
os.environ["BIRD_DB_ROOT"] = "/content/bird/dev_databases"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"
os.environ["REPO"] = "/content/claimdb-specialist-sql"
print({k: os.environ[k] for k in ["JUDGE", "NAME", "BUILDERS", "CLAIMS", "RESULTS"]})

## 2. Mount Google Drive (click Allow in the popup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(os.environ["RESULTS"], exist_ok=True)

## 3. Check the GPU

In [ ]:
!nvidia-smi

## 4. Install and start Ollama

In [ ]:
%%bash
apt-get -qq install -y zstd pciutils > /dev/null
curl -fsSL https://ollama.com/install.sh | sh > /dev/null
nohup ollama serve > /content/ollama.log 2>&1 &
until curl -s localhost:11434/api/version; do sleep 1; done; echo
ollama --version

## 5. Get the code

In [ ]:
%%bash
if [ -d "$REPO" ]; then git -C "$REPO" pull -q; else git clone -q https://github.com/Kushagra-Agnihotri/claimdb-specialist-sql.git "$REPO"; fi
pip install -q -r "$REPO/Navtej/requirements.txt"
git -C "$REPO" log --oneline -1

## 6. BIRD dev databases

The first time, this downloads BIRD (about 330 MB) and keeps a copy of the databases zip on Drive.
After that (and in the other tabs) it just unzips the Drive copy. Run this cell in Tab 1 first and wait for it
to finish before running it in the other tabs.

In [ ]:
%%bash
set -e
mkdir -p "$DRIVE/bird" /content/bird
if [ ! -f "$DRIVE/bird/dev_databases.zip" ]; then
  curl -L --fail -o /content/dev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip
  unzip -q -o /content/dev.zip -d /content/bird_raw
  cp "$(find /content/bird_raw -name dev_databases.zip | head -1)" "$DRIVE/bird/dev_databases.zip"
fi
[ -f "$BIRD_DB_ROOT/formula_1/formula_1.sqlite" ] || unzip -q -o "$DRIVE/bird/dev_databases.zip" -d /content/bird
rm -rf /content/bird/__MACOSX
ls "$BIRD_DB_ROOT"
sqlite3 "$BIRD_DB_ROOT/formula_1/formula_1.sqlite" "SELECT count(*) FROM drivers;" 2>/dev/null || python -c "import sqlite3,os;print(sqlite3.connect(os.environ['BIRD_DB_ROOT']+'/formula_1/formula_1.sqlite').execute('select count(*) from drivers').fetchone())"

## 7. Download the models

In [ ]:
%%bash
ollama pull "$JUDGE" | tail -1
for b in $BUILDERS; do ollama pull "${b%%|*}" | tail -1; done
ollama list

## 8. Speed test (20 claims, pipeline 2 with the first builder)

Prints seconds per claim and a rough total for this tab. Stop here and check before the full runs.

In [ ]:
%%bash
cd "$REPO/Navtej"
head -n 20 "$CLAIMS" > /content/speed_20.jsonl
b="${BUILDERS%% *}"
./pipeline2_C/run_C.sh --model "$JUDGE" --name "$NAME" --builder "${b%%|*}" --builder-name "${b##*|}" \
  --claims /content/speed_20.jsonl --db-root "$BIRD_DB_ROOT" --results-root "$DRIVE/speedtest" --once
python - <<'EOF'
import json, os, glob
f = glob.glob(os.path.join(os.environ["DRIVE"], "speedtest/pipeline2_C", os.environ["NAME"], "*", "speed_20", "results.jsonl"))[0]
r = [json.loads(l) for l in open(f)]
s = sum(d["seconds"] for d in r) / len(r)
runs = 1 + len(os.environ["BUILDERS"].split())
print(f"{len(r)} claims, {s:.1f} s/claim -> about {s*1000/3600:.1f} h per 1,000-claim run, "
      f"about {s*1000*runs/3600:.1f} h for this tab's {runs} runs (pipeline 1 may be slower)")
EOF

## 9. Full runs (pipeline 1, then pipeline 2 with each builder)

In [ ]:
%%bash
cd "$REPO/Navtej"
./pipeline1_A/run_A.sh --model "$JUDGE" --name "$NAME" --claims "$CLAIMS" --db-root "$BIRD_DB_ROOT" --results-root "$RESULTS"
for b in $BUILDERS; do
  ./pipeline2_C/run_C.sh --model "$JUDGE" --name "$NAME" --builder "${b%%|*}" --builder-name "${b##*|}" \
    --claims "$CLAIMS" --db-root "$BIRD_DB_ROOT" --results-root "$RESULTS"
done

## 10. Score

In [ ]:
%%bash
cd "$REPO/Navtej"
./pipeline1_A/score_A.sh --name "$NAME" --claims "$CLAIMS" --results-root "$RESULTS"
for b in $BUILDERS; do
  ./pipeline2_C/score_C.sh --name "$NAME" --builder-name "${b##*|}" --claims "$CLAIMS" --results-root "$RESULTS"
done

## Check progress (optional, any time)

In [ ]:
%%bash
for f in $(find "$RESULTS" -path "*/$NAME/*" -name results.jsonl); do echo "$(wc -l < "$f") / 1000  $f"; done